[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/05_efficientnet_b4.ipynb)

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Google Colab ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Estrategia 1: acceso por ID de carpeta raíz del proyecto
    _p1 = "/content/drive/.shortcut-targets-by-id/13VEeA8Jt0G_NNpJelWZrhacBhA_mRpnU"
    # Estrategia 2: acceso por ID de TrainData (subir un nivel)
    _p2 = "/content/drive/.shortcut-targets-by-id/13Dj5DYGOwgMGomMC8gCySKGr0ne8hOpU"

    if os.path.exists(_p1) and os.path.isdir(_p1):
        DATA_ROOT_OVERRIDE = _p1
        print(f"Estrategia 1 OK: {_p1}")
    elif os.path.exists(_p2) and os.path.isdir(_p2):
        DATA_ROOT_OVERRIDE = str(Path(_p2).parent)
        print(f"Estrategia 2 OK: {DATA_ROOT_OVERRIDE}")
    else:
        import subprocess
        _r = subprocess.run(
            ["find", "/content/drive/MyDrive", "-name", "TrainData",
             "-type", "d", "-maxdepth", "6"],
            capture_output=True, text=True, timeout=30
        )
        _hits = [p.replace('/TrainData', '') for p in _r.stdout.strip().splitlines() if p]
        if _hits:
            DATA_ROOT_OVERRIDE = _hits[0]
            print(f"Estrategia 3 OK: {DATA_ROOT_OVERRIDE}")
        else:
            print("No se encontró el dataset. Ajusta DATA_ROOT_OVERRIDE manualmente:")
            print("  DATA_ROOT_OVERRIDE = 'ruta/en/tu/Drive'")

    if DATA_ROOT_OVERRIDE:
        print(f"Dataset: {DATA_ROOT_OVERRIDE}")
else:
    print('Entorno local — se usará detección automática de rutas.')


# 05 — Fine-tuning EfficientNet-B4

EfficientNet-B4 aplica escalado compuesto (depth=1.8, width=1.4, resolution=1.3) para mayor eficiencia que ResNet-50.

---

In [ ]:
import os, sys
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL DATASET
# Si la celda de Drive de arriba encontró el dataset, DATA_ROOT_OVERRIDE
# ya está definida. Si no, puedes ajustarla manualmente aquí:
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/landslide4sense"
# ══════════════════════════════════════════════════════════════════════════════

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Drive
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/drive/MyDrive/landslide4sense"),
        Path("/content/drive/MyDrive/Landslide_ML"),
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    print("""
⚠️  Dataset no encontrado. Opciones:

━━━  OPCIÓN A — Google Drive (recomendado)  ━━━
  Ejecuta la celda de Drive de arriba y vuelve a correr esta celda.

━━━  OPCIÓN B — Kaggle API  ━━━
  !pip install kaggle -q
  # Sube tu kaggle.json y ejecuta:
  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d landslide4sense/competition -p /content/
  !unzip -q /content/competition.zip -d /content/landslide4sense/
  DATA_ROOT_OVERRIDE = "/content/landslide4sense"
""")
    raise FileNotFoundError("Dataset no encontrado. Establece DATA_ROOT_OVERRIDE con la ruta correcta.")

# ── Verificar estructura del dataset ─────────────────────────────────────────
n_train = len(list((DATA_ROOT / "TrainData" / "img").glob("*.h5")))
print(f"✓ Dataset encontrado en: {DATA_ROOT}")
print(f"  TrainData/img : {n_train} archivos .h5")


## 1. Verificar EfficientNet-B4 con timm

In [ ]:
from src.models import build_efficientnet_b4, model_summary
import torch

model = build_efficientnet_b4(n_channels=14, pretrained=True)
print(model_summary(model, n_channels=14))

dummy = torch.randn(2, 14, 128, 128)
out   = model(dummy)
print(f"Forward pass: (2,14,128,128) → {out.shape}")

## 2. Comparar EfficientNet vs ResNet-50

In [ ]:
from src.models import build_resnet50, build_efficientnet_b4

resnet = build_resnet50(n_channels=14, pretrained=False)
effnet = build_efficientnet_b4(n_channels=14, pretrained=False)

resnet_params = sum(p.numel() for p in resnet.parameters())
effnet_params = sum(p.numel() for p in effnet.parameters())

print(f"{'Modelo':<25} {'Parámetros':>15}")
print(f"{'─'*42}")
print(f"{'ResNet-50':<25} {resnet_params:>15,}")
print(f"{'EfficientNet-B4':<25} {effnet_params:>15,}")
print(f"\nEfficientNet-B4 tiene {100*(1 - effnet_params/resnet_params):.1f}% menos parámetros.")

## 3. Entrenamiento K-Fold

In [ ]:
# ⚠️ Este paso puede tomar 2-4 horas en GPU.
summary = run_kfold(cfg)
print("\nEntrenamiento EfficientNet-B4 completado.")

## 4. Comparar con ResNet-50

In [ ]:
import json
from pathlib import Path

for name, path in [("ResNet-50", "../results/resnet50"), ("EfficientNet-B4", "../results/efficientnet_b4")]:
    sp = Path(path) / "kfold_summary.json"
    if sp.exists():
        with open(sp) as f: s = json.load(f)
        agg = s["aggregated"]
        f1  = agg.get("f1", {})
        auc = agg.get("auc_roc", {})
        print(f"{name:20s}  F1={f1.get('mean',0):.4f}±{f1.get('std',0):.4f}  AUC={auc.get('mean',0):.4f}±{auc.get('std',0):.4f}")